# 1. Imports & Load Data

In [43]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# Load cleaned dataset
df = pd.read_csv("../data/processed/final_analytics_dataset.csv")

# Convert date
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

# 2. Hypothesis Testing

In [44]:
# Split groups
fast_delivery = df[df['delivery_time_days'] <= 5]['review_score'].dropna()
slow_delivery = df[df['delivery_time_days'] > 5]['review_score'].dropna()

# T-test
t_stat, p_value = ttest_ind(fast_delivery, slow_delivery)

print("T-Statistic:", t_stat)
print("P-Value:", p_value)

T-Statistic: 34.31720185774162
P-Value: 8.760478938824867e-257


If p-value < 0.05, delivery delays significantly reduce customer satisfaction.

# 3. RFM Segmentation

In [45]:
snapshot_date = df['order_purchase_timestamp'].max()

rfm = df.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'nunique',
    'total_order_value': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']

## RFM Scoring

In [46]:
rfm['R_score'] = pd.qcut(rfm['Recency'], 4, labels=[4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1,2,3,4])
rfm['M_score'] = pd.qcut(rfm['Monetary'], 4, labels=[1,2,3,4])

rfm['RFM_Score'] = (
    rfm['R_score'].astype(str) +
    rfm['F_score'].astype(str) +
    rfm['M_score'].astype(str)
)

## Segmentation

In [47]:
def segment(row):
    if row['RFM_Score'] == '444':
        return 'Champions'
    elif row['F_score'] == 4:
        return 'Loyal Customers'
    elif row['R_score'] == 1:
        return 'At Risk'
    else:
        return 'Others'

rfm['Segment'] = rfm.apply(segment, axis=1)

# 4. Churn Analysis

In [48]:
rfm['Churn_Risk'] = rfm['Recency'].apply(lambda x: 'High' if x > 90 else 'Low')

rfm['Customer_Type'] = rfm['Frequency'].apply(
    lambda x: 'Repeat' if x > 1 else 'One-Time'
)

# 5. KPI Computation (FINAL SECTION 🔥)

## KPI Framework

In [49]:
# KPI 1: Revenue Leakage Rate
cancelled_revenue = df[df['order_status'] == 'canceled']['total_order_value'].sum()
total_revenue = df['total_order_value'].sum()

revenue_leakage = (cancelled_revenue / total_revenue)
print("Revenue Leakage %:", revenue_leakage)

Revenue Leakage %: 0.006802243316677433


In [50]:
# KPI 2: Customer Lifetime Value
clv = df.groupby('customer_unique_id')['total_order_value'].sum()
avg_clv = clv.mean()

print("Average CLV:", avg_clv)

Average CLV: 174.4260249423601


In [51]:
# KPI 3: Repeat Purchase Rate
customer_orders = df.groupby('customer_unique_id')['order_id'].nunique()

repeat_customers = (customer_orders > 1).sum()
total_customers = customer_orders.count()

repeat_rate = (repeat_customers / total_customers)

print("Repeat Purchase Rate:", repeat_rate)

Repeat Purchase Rate: 0.030528191154894153


In [52]:
# KPI 4: Average Order Value
aov = df['total_order_value'].mean()
print("Average Order Value:", aov)

Average Order Value: 140.67898994167865


In [53]:
# KPI 5: Churn Risk Rate
churn_rate = (rfm['Churn_Risk'] == 'High').mean()
print("Churn Risk %:", churn_rate)

Churn Risk %: 0.8099245441207295


In [54]:
# KPI 6: RFM Segment Distribution
segment_dist = rfm['Segment'].value_counts(normalize=True) * 100
print(segment_dist)

Segment
Others             56.178998
Loyal Customers    23.095787
At Risk            18.821002
Champions           1.904213
Name: proportion, dtype: float64


In [55]:
# KPI 7: Category Revenue Contribution
category_revenue = df.groupby('product_category_name')['total_order_value'].sum()
category_share = (category_revenue / category_revenue.sum()) * 100

print(category_share.sort_values(ascending=False).head(10))

product_category_name
beleza_saude              8.960718
relogios_presentes        8.164309
cama_mesa_banho           7.976949
esporte_lazer             7.241152
informatica_acessorios    6.635303
moveis_decoracao          5.740102
utilidades_domesticas     4.948551
cool_stuff                4.522437
automotivo                4.292499
ferramentas_jardim        3.757495
Name: total_order_value, dtype: float64


In [56]:
# KPI 8: Average Delivery Time
avg_delivery_time = df['delivery_time_days'].mean()
print("Avg Delivery Time:", avg_delivery_time)

Avg Delivery Time: 12.022588617548953


In [57]:
# KPI 9: Customer Satisfaction Score
avg_rating = df['review_score'].mean()
print("Average Rating:", avg_rating)

Average Rating: 4.031389561245014


## KPI SUMMARY (🔥 FINAL OUTPUT)

In [60]:
kpi_summary = pd.DataFrame({
    "KPI": [
        "Revenue Leakage %",
        "Repeat Purchase %",
        "Average Order Value",
        "Churn Risk %",
        "Average CLV",
        "Avg Delivery Time",
        "Customer Satisfaction"
    ],
    "Value": [
        revenue_leakage,
        repeat_rate,
        aov,
        churn_rate,
        avg_clv,
        avg_delivery_time,
        avg_rating
    ]
})

print(kpi_summary)

                     KPI       Value
0      Revenue Leakage %    0.006802
1      Repeat Purchase %    0.030528
2    Average Order Value  140.678990
3           Churn Risk %    0.809925
4            Average CLV  174.426025
5      Avg Delivery Time   12.022589
6  Customer Satisfaction    4.031390


## SAVE FOR TABLEAU

In [61]:
kpi_summary.to_csv("../data/processed/kpi_summary.csv", index=False)
rfm.to_csv("../data/processed/rfm_table.csv")